<a href="https://colab.research.google.com/github/hridayrjain/BIS-Lab/blob/main/BIS1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Hriday R Jain

1BM24CS119

Algorithm : Gene Algorithm for optimization problem




Application : Cyber Attack detection Optimization



Select features and tune model parameters for network intrusion detection



Objective : Improve detection performance by reducing false alarms and computation cost


In [3]:
import numpy as np
import random
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix

X, y = make_classification(n_samples=2000, n_features=30, n_informative=15,
                           n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

NUM_FEATURES = X_train.shape[1]
PARAM_BITS = 6
CHROMOSOME_LENGTH = NUM_FEATURES + PARAM_BITS

POP_SIZE = 20
GENERATIONS = 10
MUTATION_RATE = 0.05
CROSSOVER_RATE = 0.8

def decode_chromosome(chromosome):
    feature_mask = chromosome[:NUM_FEATURES].astype(bool)
    param_bits = chromosome[NUM_FEATURES:]

    n_est_bits = "".join(str(int(b)) for b in param_bits[:3])
    n_estimators = 10 + int(n_est_bits, 2) * 20

    depth_bits = "".join(str(int(b)) for b in param_bits[3:])
    max_depth = 3 + int(depth_bits, 2) * 3

    return feature_mask, n_estimators, max_depth

def calculate_fitness(chromosome):
    feature_mask, n_estimators, max_depth = decode_chromosome(chromosome)

    if not any(feature_mask):
        return 0.0

    X_tr_selected = X_train[:, feature_mask]
    X_ts_selected = X_test[:, feature_mask]

    clf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, n_jobs=-1)
    clf.fit(X_tr_selected, y_train)
    predictions = clf.predict(X_ts_selected)

    f1 = f1_score(y_test, predictions, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()

    far = fp / (fp + tn) if (fp + tn) > 0 else 1.0
    feature_ratio = np.sum(feature_mask) / NUM_FEATURES

    fitness = (0.6 * f1) + (0.2 * (1 - far)) + (0.2 * (1 - feature_ratio))
    return fitness

def initialize_population():
    return np.random.randint(2, size=(POP_SIZE, CHROMOSOME_LENGTH))

def selection(population, fitnesses):
    total_fit = np.sum(fitnesses)
    if total_fit == 0:
        probs = np.ones(POP_SIZE) / POP_SIZE
    else:
        probs = fitnesses / total_fit
    selected_indices = np.random.choice(POP_SIZE, size=POP_SIZE, p=probs)
    return population[selected_indices]

def crossover(parent1, parent2):
    if random.random() < CROSSOVER_RATE:
        point = random.randint(1, CHROMOSOME_LENGTH - 1)
        child1 = np.concatenate((parent1[:point], parent2[point:]))
        child2 = np.concatenate((parent2[:point], parent1[point:]))
        return child1, child2
    return parent1.copy(), parent2.copy()

def mutate(chromosome):
    for i in range(CHROMOSOME_LENGTH):
        if random.random() < MUTATION_RATE:
            chromosome[i] = 1 - chromosome[i]
    return chromosome

population = initialize_population()
best_chromosome = None
best_fitness = -1

print(f"{'Gen':<6}{'Best Fitness':<15}{'Features Selected':<20}{'n_estimators':<15}{'max_depth'}")
print("-" * 65)

for generation in range(GENERATIONS):
    fitnesses = np.array([calculate_fitness(ind) for ind in population])

    max_idx = np.argmax(fitnesses)
    if fitnesses[max_idx] > best_fitness:
        best_fitness = fitnesses[max_idx]
        best_chromosome = population[max_idx].copy()

    feat_mask, n_est, depth = decode_chromosome(population[max_idx])
    print(f"{generation+1:<6}{fitnesses[max_idx]:<15.4f}{np.sum(feat_mask):<20}{n_est:<15}{depth}")

    next_population = selection(population, fitnesses)

    for i in range(0, POP_SIZE, 2):
        if i+1 < POP_SIZE:
            child1, child2 = crossover(next_population[i], next_population[i+1])
            next_population[i] = mutate(child1)
            next_population[i+1] = mutate(child2)

    population = next_population

best_features, best_n_est, best_depth = decode_chromosome(best_chromosome)
print("\n" + "="*40 + "\nOPTIMIZATION COMPLETE\n" + "="*40)
print(f"Optimal Fitness Score: {best_fitness:.4f}")
print(f"Features Retained: {np.sum(best_features)} out of {NUM_FEATURES}")
print(f"Optimized Hyperparameters: n_estimators={best_n_est}, max_depth={best_depth}")


Gen   Best Fitness   Features Selected   n_estimators   max_depth
-----------------------------------------------------------------
1     0.7677         18                  90             18
2     0.7762         13                  90             15
3     0.8003         15                  110            24
4     0.7944         15                  110            24
5     0.7828         16                  30             18
6     0.7939         12                  50             12
7     0.7885         15                  30             24
8     0.7796         14                  150            24
9     0.7686         17                  110            12
10    0.7832         14                  110            24

OPTIMIZATION COMPLETE
Optimal Fitness Score: 0.8003
Features Retained: 15 out of 30
Optimized Hyperparameters: n_estimators=110, max_depth=24
